In [1]:
import math
import numpy as np  # usado apenas para operacoes vetoriais (arrays de nos/pesos), conforme permitido pelo enunciado

In [2]:
def func(x, y, item):
    '''
    Funcao que recebe as variaveis x e y e retorna o resultados para cada
    exemplo do EP
    --> x,y  = variaveis da funcao
    --> item = exemplo escolhido pelo usuario

    Observacao: para a maioria dos exemplos a integral dupla original foi
    reduzida analiticamente (integrando-se exatamente em uma das variaveis)
    a uma unica integral, que e entao calculada numericamente pela formula
    de Gauss. Isso e valido: a quadratura de Gauss so precisa ser aplicada
    numericamente na(s) variavel(is) que restarem apos a reducao exata, sem
    perda de precisao. Para o exemplo 3.1, que nao possui primitiva
    elementar, a integral dupla e de fato calculada numericamente nas duas
    variaveis (ver funcao integr_var).
    '''
    if item == '1.1':
        # Volume do cubo de aresta 1: f(x,y,z) = 1. Integrando exatamente em
        # y e z (ambos de 0 a 1) sobra f(x) = 1.
        return 1
    if item == '1.2':
        # Volume do tetraedro de vertices (0,0,0),(1,0,0),(0,1,0),(0,0,1):
        # a altura e z = 1 - x - y, com 0 <= y <= 1 - x. Integrando
        # exatamente em y: int_0^(1-x) (1-x-y) dy = (1-x)^2/2, que
        # expandido da (x^2)/2 - x + 1/2.
        return (x**2)/2 - x + 1/2
    if item == '2.1':
        # Area sob y = 1 - x^2: integrando a funcao constante 1 em y de 0 a
        # 1-x^2 sobra exatamente o integrando 1 - x^2.
        return 1-x**2
    if item == '2.2':
        # Mesma area do item anterior, mas calculada integrando primeiro em
        # x (de 0 a sqrt(1-y)): integrando constante 1 em x sobra sqrt(1-y)
        # (aqui o parametro "x" representa, na verdade, y).
        return math.sqrt(1-x)
    if item == '3.1':
        # Elemento de area da superficie z = e^(y/x):
        # sqrt(fx^2 + fy^2 + 1), com fx = -(y/x^2)*e^(y/x) e
        # fy = (1/x)*e^(y/x). Nao ha reducao analitica possivel aqui (sem
        # primitiva elementar), entao a integral dupla e calculada
        # numericamente nas duas variaveis (ver integr_var).
        return math.sqrt((x**2+y**2)*math.e**(2*y/x)+x**4)/(x**2)
    if item == '3.2':
        # Volume sob z = e^(y/x), com x^3 <= y <= x^2: integrando
        # exatamente em y, int_{x^3}^{x^2} e^(y/x) dy = x*(e^x - e^(x^2)),
        # logo o integrando (em x) e -x*(e^(x^2) - e^x).
        return -x*(math.e**(x**2)-math.e**x)
    if item == '4.1':
        # Calota esferica de altura 1/4 (raio 1), obtida por solido de
        # revolucao em torno do eixo y: d_gamma(x,y) = x (distancia ao
        # eixo). Integrando exatamente em y de 3/4 a sqrt(1-x^2):
        # x*(sqrt(1-x^2) - 3/4), multiplicado por 2*pi.
        return 2*math.pi*x*(math.sqrt(1-x**2) - 3/4)
    if item == '4.2':
        # Solido de revolucao em torno do eixo y da regiao
        # 0 <= x <= e^(-y^2), -1 <= y <= 1: integrando exatamente em x
        # (a distancia ao eixo e o proprio x), int_0^(e^(-y^2)) x dx =
        # (e^(-y^2))^2/2, multiplicado por 2*pi. Aqui o parametro "x"
        # representa, na verdade, y.
        return math.pi*(math.e**(-x**2))**2

In [3]:
def interp(vx, vw, m, M):
    '''
    Funcao que recebe um vetor de nos e pesos (no intervalo padrao [-1,1])
    e os interpola devidamente para o intervalo [m, M] por mudanca de
    variavel linear.
    --> vx = vetor de nos (em [-1,1])
    --> vw = vetor de pesos (em [-1,1])
    --> m  = intervalo inferior
    --> M  = intervalo superior
    '''
    # Mudanca de variavel linear x = ((M-m)/2)*t + (M+m)/2, com t em
    # [-1,1], mapeia o intervalo padrao [-1,1] no intervalo [m,M].
    x_ret = []
    for i in range(len(vx)):
        x_ret.append(vx[i]*(M-m)/2 + (M+m)/2)
    x_ret = np.array(x_ret)

    # O jacobiano dessa mudanca de variavel, dx/dt = (M-m)/2, multiplica os
    # pesos originais (fator de escala mencionado no enunciado).
    vw = np.array(vw)
    w_ret = vw*(M-m)/2

    return x_ret, w_ret

In [4]:
def integr(s, nox, noy, coefx, coefy, item):
    '''
    Formula de quadratura de Gauss com s nos.
    --> s            = numero de nos (6, 8 ou 10)
    --> nox, noy     = vetor de nos de x e, se houver, de y
    --> coefx, coefy = vetor de pesos de x e, se houver, de y
    --> item = exemplo escolhido pelo usuario

    Se noy == 0, calcula a integral simples (1 variavel):
        I = sum_i coefx[i] * f(nox[i])
    Caso contrario, calcula a integral dupla em um retangulo FIXO
    [a,b] x [c,d] (os mesmos nos/pesos de y valem para todo i):
        I = sum_i coefx[i] * [ sum_j coefy[j] * f(nox[i], noy[j]) ]
    Para integrais duplas com limites de y variaveis (dependentes de x,
    como no exemplo 3.1), use a funcao integr_var, pois nesse caso os
    nos/pesos de y mudam a cada no xi.
    '''
    if isinstance(noy, int) and noy == 0:
        # Formula de Gauss 1D: aproxima a integral pela soma ponderada do
        # integrando nos nos de Gauss (ja transportados para [a,b]).
        A = 0
        for i in range(s):
            A += coefx[i]*func(nox[i], 0, item)
    else:
        # Formula de Gauss iterada (dupla) em retangulo fixo: para cada no
        # xi, calcula-se F(xi) = sum_j coefy[j]*f(xi, yj); depois soma-se
        # em i ponderando por coefx[i].
        A = 0
        for i in range(s):
            F = 0
            for j in range(s):
                F += coefy[j]*func(nox[i], noy[j], item)
            A += coefx[i]*F
    return A

In [5]:
def integr_var(s, nox, coefx, raw_nodes, raw_w, c, d, item):
    '''
    Formula de Gauss iterada (dupla) para o caso em que os limites de
    integracao em y DEPENDEM de x, isto e, y varia em [c(xi), d(xi)] para
    cada no xi (regiao do tipo {(x,y) | a<=x<=b, c(x)<=y<=d(x)}, como no
    exemplo 3.1, em que c(x) = x^3 e d(x) = x^2).

    Como os limites de y mudam a cada no xi, os nos e pesos de y NAO podem
    ser calculados uma unica vez para todos os i (diferente do caso de
    retangulo fixo em integr): eles precisam ser recalculados, via mudanca
    de variavel (funcao interp), para cada xi, a partir dos nos/pesos
    "crus" no intervalo padrao [-1,1] (raw_nodes, raw_w).

    --> s                = numero de nos
    --> nox, coefx       = nos e pesos de x, ja transportados para [a,b]
    --> raw_nodes, raw_w = nos e pesos de Gauss no intervalo padrao [-1,1]
    --> c, d             = funcoes que dao os limites inferior/superior de
                           y em funcao de x: c(x) e d(x)
    --> item             = exemplo escolhido pelo usuario
    '''
    A = 0
    for i in range(s):
        xi = nox[i]
        # Para este xi, os limites de y sao c(xi) e d(xi): recalcula os
        # nos/pesos de y especificamente para este intervalo (formula
        # iterada: yij e vij dependem de xi, como descrito no enunciado).
        noy_i, coefy_i = interp(raw_nodes, raw_w, c(xi), d(xi))
        F = 0
        for j in range(s):
            F += coefy_i[j]*func(xi, noy_i[j], item)
        A += coefx[i]*F
    return A

In [6]:
def main():
    '''
    Funcao principal que imprime o numero de nos escolhido pelo usuario e
    o valor da integral calculada para cada caso.

    Pede-se para o usuário digitar o caso escolhido de acordo com uma lista e
    o numero de nos utilizados no calculo das integrais
    '''
    # Nos e pesos de Gauss-Legendre tabelados no intervalo padrao [-1,1],
    # para n = 6, 8 e 10 (ja escritos de forma simetrica em torno de 0,
    # conforme a propriedade de paridade mencionada no enunciado).
    x1 =    [-0.9324695142031520278123016, -0.6612093864662645136613996, -0.2386191860831969086305017,
              0.2386191860831969086305017,  0.6612093864662645136613996,  0.9324695142031520278123016]
    w1 =    [ 0.1713244923791703450402961,  0.3607615730481386075698335,  0.4679139345726910473898703,
              0.4679139345726910473898703,  0.3607615730481386075698335,  0.1713244923791703450402961]
    
    x2 =    [-0.9602898564975362316835609, -0.7966664774136267395915539,
             -0.5255324099163289858177390, -0.1834346424956498049394761,
              0.1834346424956498049394761,  0.5255324099163289858177390, 
              0.7966664774136267395915539,  0.9602898564975362316835609]
    w2 =    [ 0.1012285362903762591525314,  0.2223810344533744705443560,
              0.3137066458778872873379622,  0.3626837833783619829651504,
              0.3626837833783619829651504,  0.3137066458778872873379622,
              0.2223810344533744705443560,  0.1012285362903762591525314]
    
    x3 =    [-0.9739065285171717200779640, -0.8650633666889845107320967, -0.6794095682990244062343274,
             -0.4333953941292471907992659, -0.1488743389816312108848260,
              0.1488743389816312108848260,  0.4333953941292471907992659,  0.6794095682990244062343274,
              0.8650633666889845107320967,  0.9739065285171717200779640]
    w3 =    [ 0.0666713443086881375935688,  0.1494513491505805931457763,  0.2190863625159820439955349,
              0.2692667193099963550912269,  0.2955242247147528701738930,
              0.2955242247147528701738930,  0.2692667193099963550912269,  0.2190863625159820439955349,
              0.1494513491505805931457763,  0.0666713443086881375935688]

    ex = str(input('''Digite o item do exemplo de acordo:
    1.1 -- Volume do cubo de aresta com compr. 1
    1.2 -- Volume do tetraedro de vertices dados
    2.1 -- Area da regiao do 1o quadr. abaixo da funcao 1-x^2
    2.2 -- Area da regiao do 1o quadr. abaixo da funcao (1-y)^1/2
    3.1 -- Area da superficie da funcao z = e^(y/x)
    3.2 -- Volume abaixo da funcao z = e^(y/x)
    4.1 -- Volume da calota esferica
    4.2 -- Volume do solido de revolucao 
    '''))

    # O enunciado exige testar com n = 6, 8 e 10 nos, e pede que os
    # parametros do teste sejam dados como entrada pelo usuario (sem editar
    # o codigo). Por isso o numero de nos e perguntado aqui, com validacao.
    n = int(input('Digite o numero de nos da integracao de Gauss [6, 8 ou 10]: '))
    while n not in (6, 8, 10):
        n = int(input('Valor invalido. Digite 6, 8 ou 10: '))

    if n == 6:
        x = x1.copy()
        y = x1.copy()
        w = w1.copy()
    elif n == 8:
        x = x2.copy()
        y = x2.copy()
        w = w2.copy()
    elif n == 10:
        x = x3.copy()
        y = x3.copy()
        w = w3.copy()

    if ex == '1.1' or ex == '1.2' or ex == '2.1' or ex == '2.2':
        if ex == '1.1':
            print('Volume do cubo de aresta com compr. 1')
            print('funcao: f(x) = 1', end = '')
        if ex == '1.2':
            print('Volume do tetraedro de vertices dados')
            print('funcao: f(x) = (x^2)/2 - x + 1/2', end = '')
        if ex == '2.1':
            print('Area da regiao do 1o quadr. abaixo da funcao 1-x^2')
            print('funcao: f(x) = 1 - x^2', end = '')
        if ex == '2.2':
            print('Area da regiao do 1o quadr. abaixo da funcao (1-y)^1/2')
            print('funcao: f(x) = sqrt(1-x)', end = '')
        # Nestes 4 exemplos a integral dupla foi reduzida analiticamente a
        # uma integral simples (ver comentarios em func()); basta aplicar
        # a formula de Gauss 1D no intervalo [a,b].
        a, b = 0, 1
        print(f', {a} <= x <= {b}')
        x, wx = interp(x, w, a, b)
        I = integr(n, x, 0, wx, 0, ex)

    if ex == '3.1':
        print('Area da superficie da funcao z = e^(y/x)')
        print('funcao: f(x,y) = sqrt((x^2+y^2)*e^(2*y/x)+x^4)/(x^2)')
        a, b = 0.1, 0.5
        print(f'{a} <= x <= {b}, x^3 <= y <= x^2')
        # Guarda os nos/pesos "crus" (intervalo padrao [-1,1]) ANTES de
        # transformar x para [a,b]: serao reaproveitados para recalcular os
        # nos/pesos de y em cada xi, ja que os limites de y (x^3 e x^2)
        # dependem de x e por isso NAO podem usar um unico intervalo fixo
        # igual para todos os nos (erro presente na versao anterior, que
        # usava c,d = 0.001, 0.25 fixos para todo i).
        raw_nodes, raw_w = x.copy(), w.copy()
        x, wx = interp(x, w, a, b)
        I = integr_var(n, x, wx, raw_nodes, raw_w,
                        lambda xv: xv**3, lambda xv: xv**2, ex)

    if ex == '3.2':
        print('Volume abaixo da funcao z = e^(y/x)')
        print('funcao: f(x) = -x(e^(x^2)-e^x)', end='')
        # Aqui a integracao exata em y (de x^3 a x^2) ja foi feita
        # analiticamente dentro de func(), restando apenas a integral em x.
        a, b = 0.1, 0.5
        print(f', {a} <= x <= {b}')
        x, wx = interp(x, w, a, b)
        I = integr(n, x, 0, wx, 0, ex)

    if ex == '4.1':
        print('Volume da calota esferica')
        print('funcao: f(x) = 2pi*x*(sqrt(1-x^2) - 3/4)', end='')
        # Limite superior de x e o ponto onde sqrt(1-x^2) = 3/4 (topo da
        # calota, x = sqrt(7)/4).
        a, b = 0, math.sqrt(7)/4
        print(f', {a} <= x <= %.2f'%b)
        x, wx = interp(x, w, a, b)
        I = integr(n, x, 0, wx, 0, ex)

    if ex == '4.2':
        print('Volume do solido de revolucao')
        print('funcao: f(y) = pi*(e^(-y^2))^2, -1 <= y <= 1')
        # O intervalo de integracao (-1 a 1) coincide com o intervalo
        # padrao dos nos de Gauss, entao nao e necessario chamar interp().
        wx = w.copy()
        I = integr(n, x, 0, wx, 0, ex)

    print(f'Para n = {n}, o valor da integral calculada será {I}')

main()

Volume abaixo da funcao z = e^(y/x)
funcao: f(x) = -x(e^(x^2)-e^x), 0.1 <= x <= 0.5
Para n = 8, o valor da integral calculada será 0.033305566116232074
